<a href="https://colab.research.google.com/github/AndreaGhioldi03/milan-airbnb-price-analysis/blob/main/01_milan_airbnb_analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Milan Airbnb Price Analysis

**Research question:** How much does it cost to stay in Milan on Airbnb, and what drives the price?

1. How do prices vary across neighbourhoods?
2. How much do room type, guest capacity and reviews matter?
3. How do professional hosts (many listings) differ from private ones?

**Data source:** Inside Airbnb (Milan, detailed listings)

## Data loading

In [ ]:
import pandas as pd

url = "https://data.insideairbnb.com/italy/lombardy/milan/2026-09-19/data/listings.csv.gz"
listings = pd.read_csv(url)

print(listings.shape)
listings.head()


(25338, 90)


,id,listing_url,scrape_id,last_scraped,source,name,description,neighborhood_overview,picture_url,host_id,...,review_scores_communication,review_scores_location,review_scores_value,license,instant_bookable,calculated_host_listings_count,calculated_host_listings_count_entire_homes,calculated_host_listings_count_private_rooms,calculated_host_listings_count_shared_rooms,reviews_per_month
0,23986,https://www.airbnb.com/rooms/23986,20260919134113,2026-09-21,NaN,""" Characteristic Milanese flat""",I look forward to welcoming you in my flat; it...,NaN,https://a0.muscache.com/pictures/623d63f8-56cf...,95941,...,4.73,4.69,4.46,NaN,NaN,1,1,0,0,0.15
1,40470,https://www.airbnb.com/rooms/40470,20260919134113,2026-09-20,NaN,Giacinto Cosy & clean flat near MM1,<b>The space</b><br />If you want to visit Mil...,NaN,https://a0.muscache.com/pictures/891684/01c17b...,174203,...,4.88,4.45,4.52,IT015146C2YDCLSU8G,NaN,2,2,0,0,0.23
2,46536,https://www.airbnb.com/rooms/46536,20260919134113,2026-09-20,NaN,Nico & Cinzia's Pink Suite!,Over the international fair in April we rent f...,NaN,https://a0.muscache.com/pictures/4eb8e0f5-e17b...,138683,...,4.92,4.36,4.59,IT015209C2BBKGCMT7,NaN,1,1,0,0,0.22
3,84741,https://www.airbnb.com/rooms/84741,20260919134113,2026-09-21,NaN,Nice room with private bathroom,Independent room with private bathroom.<br />C...,NaN,https://a0.muscache.com/pictures/0b34089b-4d58...,443989,...,4.89,4.18,4.64,"I applied for a CIN, but haven’t received it yet",NaN,3,1,2,0,3.75
4,88130,https://www.airbnb.com/rooms/88130,20260919134113,2026-09-21,NaN,Great comfortable & quite APT. x 4 people!,Our bright and comfortable apartment for 3+ 1 ...,NaN,https://a0.muscache.com/pictures/606790/408e10...,481475,...,5.00,4.43,4.71,NaN,NaN,1,1,0,0,0.04


I loaded the detailed listings file for Milan from Inside Airbnb.
The raw dataset contains 25338 rows and 90 columns, covering host, location, property, price and review information.

## Column selection and missing values

In [ ]:
cols = [
    "id", "price",
    "neighbourhood_cleansed", "latitude", "longitude",
    "room_type", "property_type", "accommodates", "bedrooms", "beds", "bathrooms",
    "minimum_nights",
    "number_of_reviews", "review_scores_rating", "reviews_per_month",
    "availability_365",
    "host_id", "host_is_superhost", "calculated_host_listings_count",
]

df = listings[cols].copy()

print(df.shape)
df.info()
print(df.isna().sum().sort_values(ascending=False))
df["price"].head(10)

(25338, 19)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 25338 entries, 0 to 25337
Data columns (total 19 columns):
 #   Column                          Non-Null Count  Dtype  
---  ------                          --------------  -----  
 0   id                              25338 non-null  int64  
 1   price                           21654 non-null  object 
 2   neighbourhood_cleansed          25338 non-null  object 
 3   latitude                        25338 non-null  float64
 4   longitude                       25338 non-null  float64
 5   room_type                       25338 non-null  object 
 6   property_type                   25338 non-null  object 
 7   accommodates                    25338 non-null  int64  
 8   bedrooms                        25329 non-null  float64
 9   beds                            24784 non-null  float64
 10  bathrooms                       0 non-null      float64
 11  minimum_nights                  25186 non-null  float64
 12  number_of_reviews   

,price
0,$203.37
1,$137.00
2,NaN
3,NaN
4,$102.33
5,NaN
6,$105.00
7,$101.70
8,$105.72
9,$130.00


I kept 19 of the 90 total columns that are relevant to the research question: price, location, property characteristics, reviews, availability and host profile. Links, photos and free-text descriptions are dropped.

Main findings on data quality:
- `price` is stored as text (e.g. "$203.37") and is missing for 3684 rows (14.5%).
- `bathrooms` is completely empty, so I dropped it.
- Other gaps are small (e.g. `beds`: 554, `bedrooms`: 9).
Rows without a price cannot be analysed, so they are removed in the next step.

In [ ]:
df = df.drop(columns=["bathrooms"])

df["price"] = (
    df["price"]
    .str.replace("$", "", regex=False)
    .str.replace(",", "", regex=False)
    .astype(float)
)

df = df.dropna(subset=["price"]).copy()
print(df.shape)

print("Duplicated ids:", df["id"].duplicated().sum())
print(df["price"].describe(percentiles=[0.01, 0.05, 0.25, 0.5, 0.75, 0.95, 0.99]))

print("Listings with 0 reviews:", (df["number_of_reviews"] == 0).sum())
print("Missing rating:", df["review_scores_rating"].isna().sum())

(21654, 18)
Duplicated ids: 0
count    21654.000000
mean       228.827648
std        824.133616
min         14.100000
1%          38.895300
5%          69.496500
25%        120.100000
50%        163.670000
75%        232.000000
95%        481.883000
99%       1107.000000
max      88100.000000
Name: price, dtype: float64
Listings with 0 reviews: 2931
Missing rating: 2931


I converted `price` from text to a numeric value (US dollars, as reported in the dataset) and removed rows without a price, leaving 21654 rows. There are no duplicated listing ids.

Missing review scores correspond exactly to rows with zero reviews (2931), so a missing score means "not reviewed yet". I left these values empty instead of inventing a score.

The price distribution is strongly skewed: the median is 163.67 per night, but the mean is 228.83 and the maximum is 88100. The most expensive listings are not credible (e.g. a
2-bedroom flat at 88100 per night!), probably data entry errors or prices not referring to a
single night.

**Takeaway:** a few extreme values distort the mean, so I describe prices with the median and I filter out non-comparable listings in the next step.

In [ ]:
cols_show = ["price", "room_type", "accommodates", "bedrooms", "minimum_nights", "number_of_reviews"]
print(df.sort_values("price", ascending=False)[cols_show].head(15))

print("Listings above 1000:", (df["price"] > 1000).sum())

print(df["minimum_nights"].describe(percentiles=[0.5, 0.9, 0.95, 0.99]))
print("Min nights >= 30:", (df["minimum_nights"] >= 30).sum())

print("Availability 365 = 0:", (df["availability_365"] == 0).sum())

          price        room_type  accommodates  bedrooms  minimum_nights  \
17725  88100.00  Entire home/apt             6       2.0             NaN   
15448  57765.00  Entire home/apt             2       1.0             1.0   
20441  20925.00  Entire home/apt             6       2.0             2.0   
16728  17633.00  Entire home/apt             4       2.0             1.0   
25160  14400.00  Entire home/apt             6       2.0             1.0   
17772  14070.00  Entire home/apt             4       1.0             1.0   
18730  13757.00  Entire home/apt             4       1.0             1.0   
15922  12889.00  Entire home/apt             6       2.0             1.0   
13697  12064.00  Entire home/apt             6       2.0             1.0   
12413  11628.00  Entire home/apt             4       1.0             1.0   
2537   10797.56     Private room             1       1.0            31.0   
12428  10039.00  Entire home/apt             5       1.0             1.0   
8987    9607

Before filtering, I looked at the most expensive listings and at the minimum-stay rules to decide which listings can be compared.

**Extreme prices:** 277 listings are priced above 1000 per night. The 15 most expensive range from $8150 to 88100 and are almost all ordinary apartments (14 entire homes and 1 private room, with 1-2 bedrooms and space for 1-6 guests). Size and type cannot justify such prices. Some of these listings have many reviews (up to 151), which means they have hosted guests, so the listed price is probably not what guests actually paid (e.g. placeholder or
mistyped prices). The data does not allow me to verify the cause.

**Minimum stay:** The distribution has two clear groups: the median is 1 night, while the 90th, 95th and 99th percentiles are all 31 nights. 2464 rows (about 11%) require at least 30 nights, so they are medium-term rentals rather than tourist stays. The mean (about 5.4 nights) is misleading here, since it sits between the two groups.

**Availability:** 135 listings have no availability in the next 365 days (1 year). This can mean fully booked or a blocked calendar. They are only 0.6% of the sample, so I keep them.

## Data cleaning

In [ ]:
n_start = len(df)

df = df[df["minimum_nights"] < 30]
n_after_nights = len(df)

df = df[df["price"] <= 1000].copy()
n_after_price = len(df)

print("Start:", n_start)
print("After min nights < 30:", n_after_nights, "| removed:", n_start - n_after_nights)
print("After price <= 1000:", n_after_price, "| removed:", n_after_nights - n_after_price)

print(df["price"].describe())
print(df.groupby("room_type")["price"].agg(["count", "median"]).round(1))

Start: 21654
After min nights < 30: 19055 | removed: 2599
After price <= 1000: 18827 | removed: 228
count    18827.000000
mean       201.218748
std        120.661273
min         14.400000
25%        128.930000
50%        168.000000
75%        233.000000
max       1000.000000
Name: price, dtype: float64
                 count  median
room_type                     
Entire home/apt  17016   172.3
Hotel room          15   159.0
Private room      1753   116.0
Shared room         43    46.0


**Decision:** the analysis is restricted to tourist stays (minimum stay < 30 nights), and prices above $1000 per night are treated as implausible and excluded.

The final sample has 18827 listings, with a median price of $168 per night, mean 201, and a range 14–1000): now median and mean are close, a sign that extreme values no longer dominate.

Looking at room type, entire home/apartments are the largest majority (17016 rows, about 90%) with a median of 172; private rooms (1753) cost a median of 116. Hotel rooms (15) and shared rooms (43) are too few to draw reliable conclusions, so I keep them in the dataset but without commenting them.

The $1000 threshold is a judgement call; since the analysis relies on medians, the results are robust to reasonable changes of this cutoff.